# Cross-model token cost: Turkish vs English

Same corpus as `turkish-fertility.ipynb`: OPUS-100 en-tr, test split, 2000 sentence pairs.
The metric is total tokens / total tokens (tr/en) over the whole parallel corpus.

- **Local** tokenizers are measured per sentence and summed, exactly like the first notebook.
- **API** tokenizers (Claude, Gemini) are measured by joining the 2000 sentences with `"\n"` into one text per language,
  so each language costs one request per model. The same join is measured locally with `o200k_base` as a calibration row.
- Only token counting and model listing endpoints are called. Nothing is generated. API keys are read from the repo-root `.env` (git-ignored) or the environment and never printed.

In [1]:
import os, sys
from dotenv import dotenv_values, find_dotenv

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Keys come from the repo-root .env (git-ignored). Empty values are skipped, values are never printed.
for k, v in dotenv_values(find_dotenv(usecwd=True)).items():
    if v and not os.environ.get(k):
        os.environ[k] = v

print(sys.executable)
print({k: bool(os.environ.get(k)) for k in ["ANTHROPIC_API_KEY", "GEMINI_API_KEY", "HF_TOKEN"]})

D:\GithubRepos\llm-internals\.venv\Scripts\python.exe
{'ANTHROPIC_API_KEY': True, 'GEMINI_API_KEY': True, 'HF_TOKEN': True}


In [2]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-tr", split="test")
pairs = [(r["translation"]["en"], r["translation"]["tr"]) for r in ds]
en_texts = [en for en, tr in pairs]
tr_texts = [tr for en, tr in pairs]
en_joined = "\n".join(en_texts)
tr_joined = "\n".join(tr_texts)
print(len(pairs), len(en_joined), len(tr_joined))

2000 85052 79988


## Local tokenizers

Results are collected in `rows`. Anything that cannot be measured gets a row with `None` counts and the reason in `note`.

In [3]:
import tiktoken
from transformers import AutoTokenizer

rows = []

def add(model, source, en, tr, note=""):
    rows.append({"model/tokenizer": model, "source": source, "en tokens": en, "tr tokens": tr, "note": note})

def count(tok, text):
    if hasattr(tok, "encode_ordinary"):
        return len(tok.encode_ordinary(text))
    return len(tok.encode(text, add_special_tokens=False))

def total(tok, texts):
    return sum(count(tok, t) for t in texts)

o200k = tiktoken.get_encoding("o200k_base")
add("o200k_base (reference)", "local", total(o200k, en_texts), total(o200k, tr_texts), "per-sentence sum")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
HF_MODELS = {
    "Qwen/Qwen3.8-27B": False,          # newest Qwen3-series release on the Hub (Aug 2026), not gated
    "deepseek-ai/DeepSeek-V3": False,
    "meta-llama/Llama-3.1-8B-Instruct": True,  # gated
    "google/gemma-3-4b-it": True,              # gated
}

hf_tok = {}
for repo, gated in HF_MODELS.items():
    try:
        tok = AutoTokenizer.from_pretrained(repo)
    except Exception as e:
        if not gated:
            reason = f"load failed: {type(e).__name__}"
        elif os.environ.get("HF_TOKEN"):
            reason = "gated repo, access not granted to this Hugging Face account"
        else:
            reason = "gated repo, no HF_TOKEN set"
        print(f"{repo}: skipped ({type(e).__name__})")
        add(repo, "local", None, None, f"not measured: {reason}")
        continue
    hf_tok[repo] = tok
    add(repo, "local", total(tok, en_texts), total(tok, tr_texts), f"per-sentence sum, vocab {len(tok):,}")
    print(f"{repo}: vocab {len(tok):,}")

Qwen/Qwen3.8-27B: vocab 248,077


deepseek-ai/DeepSeek-V3: vocab 128,815


meta-llama/Llama-3.1-8B-Instruct: vocab 128,256


google/gemma-3-4b-it: skipped (OSError)


## Calibration: joined text vs per-sentence sum

The API rows below send one `"\n"`-joined text per language. To see how much the join itself moves the ratio,
the same joined text is tokenized locally with `o200k_base` and compared with the per-sentence 1.217.

In [5]:
cal_en, cal_tr = count(o200k, en_joined), count(o200k, tr_joined)
add("o200k_base, joined (calibration)", "local", cal_en, cal_tr, "same \\n join as API rows")
print(cal_en, cal_tr, round(cal_tr / cal_en, 3), "vs per-sentence", round(rows[0]["tr tokens"] / rows[0]["en tokens"], 3))

20734 25196 1.215 vs per-sentence 1.217


## Claude (Anthropic API, `messages.count_tokens`)

Model IDs come from `client.models.list()`: newest Opus, newest Sonnet, and `claude-sonnet-4-6` for the older tokenizer.
`count_tokens` counts the whole user message, so each number includes a few tokens of message framing (constant, and the same for en and tr).

In [6]:
CLAUDE_TARGETS = ["latest Opus", "latest Sonnet", "claude-sonnet-4-6"]

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set, Claude rows not measured")
    for t in CLAUDE_TARGETS:
        add(f"Claude ({t})", "API", None, None, "not measured: ANTHROPIC_API_KEY not set")
else:
    import anthropic
    # Keys not scoped to a workspace need the workspace ID as a header.
    ws = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    client = anthropic.Anthropic(default_headers={"anthropic-workspace-id": ws} if ws else None)
    try:
        models = sorted(client.models.list(limit=1000), key=lambda m: m.created_at, reverse=True)
    except anthropic.APIStatusError as e:
        print("models.list failed:", e.status_code, e.message)
        models = []
        reason = "workspace ID required for this key" if "workspace" in str(e.message) else f"models.list failed ({e.status_code})"
    ids = [m.id for m in models]
    print(ids)

    def newest(prefix):
        return next((m.id for m in models if m.id.startswith(prefix)), None)

    picked = {
        "latest Opus": newest("claude-opus"),
        "latest Sonnet": newest("claude-sonnet"),
        "claude-sonnet-4-6": "claude-sonnet-4-6" if "claude-sonnet-4-6" in ids else None,
    }

    def claude_count(model, text):
        r = client.messages.count_tokens(model=model, messages=[{"role": "user", "content": text}])
        return r.input_tokens

    for label, mid in picked.items():
        if not models:
            add(f"Claude ({label})", "API", None, None, f"not measured: {reason}")
            continue
        if mid is None:
            add(f"Claude ({label})", "API", None, None, "not measured: model not in models.list()")
            continue
        try:
            add(mid, "API", claude_count(mid, en_joined), claude_count(mid, tr_joined), "\\n joined, includes message framing")
        except Exception as e:
            print(mid, type(e).__name__, e)
            add(mid, "API", None, None, f"not measured: {type(e).__name__}")

['claude-sonnet-5-5', 'claude-opus-5-5', 'claude-fable-5-1', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'claude-opus-4-8', 'claude-opus-4-7', 'claude-sonnet-4-6', 'claude-opus-4-6', 'claude-opus-4-5-20251101', 'claude-haiku-4-5-20251001', 'claude-sonnet-4-5-20250929']


## Gemini (Gemini API, `models.count_tokens`)

The model ID comes from `client.models.list()`: among models that support `countTokens`, the highest-versioned Gemini Pro model is used (falls back to the highest-versioned Gemini model).

In [7]:
import re

if not os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY not set, Gemini row not measured")
    add("Gemini", "API", None, None, "not measured: GEMINI_API_KEY not set")
else:
    from google import genai
    gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    skip = ("embedding", "tts", "image", "audio", "live", "robotics", "computer-use")
    names = [m.name.removeprefix("models/") for m in gclient.models.list()
             if "countTokens" in (m.supported_actions or []) and m.name.removeprefix("models/").startswith("gemini")
             and not any(s in m.name for s in skip)]
    print(names)

    def version(name):
        m = re.match(r"gemini-(\d+(?:\.\d+)?)", name)
        return float(m.group(1)) if m else 0.0

    pro = [n for n in names if "-pro" in n]
    pool = pro or names
    if not pool:
        add("Gemini", "API", None, None, "not measured: no Gemini model with countTokens in models.list()")
    else:
        top = max(version(n) for n in pool)
        gid = sorted(n for n in pool if version(n) == top)[0]
        try:
            g_en = gclient.models.count_tokens(model=gid, contents=en_joined).total_tokens
            g_tr = gclient.models.count_tokens(model=gid, contents=tr_joined).total_tokens
            add(gid, "API", g_en, g_tr, "\\n joined")
        except Exception as e:
            print(gid, type(e).__name__, e)
            add(gid, "API", None, None, f"not measured: {type(e).__name__}")

['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']


## Results

In [8]:
import pandas as pd

add("GPT-6", "n/a", None, None, "not measured: tokenizer not published")

df = pd.DataFrame(rows)
df["tr/en"] = [round(t / e, 3) if e and t else None for e, t in zip(df["en tokens"], df["tr tokens"])]
df = df[["model/tokenizer", "source", "en tokens", "tr tokens", "tr/en", "note"]]
for c in ["en tokens", "tr tokens"]:
    df[c] = df[c].astype("Int64")
pd.set_option("display.max_colwidth", 80)
df

,model/tokenizer,source,en tokens,tr tokens,tr/en,note
0,o200k_base (reference),local,20666,25146,1.217,per-sentence sum
1,Qwen/Qwen3.8-27B,local,21623,25687,1.188,"per-sentence sum, vocab 248,077"
2,deepseek-ai/DeepSeek-V3,local,21623,33457,1.547,"per-sentence sum, vocab 128,815"
3,meta-llama/Llama-3.1-8B-Instruct,local,21567,25141,1.166,"per-sentence sum, vocab 128,256"
4,google/gemma-3-4b-it,local,<NA>,<NA>,NaN,"not measured: gated repo, access not granted to this Hugging Face account"
5,"o200k_base, joined (calibration)",local,20734,25196,1.215,same \n join as API rows
6,claude-opus-5-5,API,32239,46089,1.430,"\n joined, includes message framing"
7,claude-sonnet-5-5,API,32239,46089,1.430,"\n joined, includes message framing"
8,claude-sonnet-4-6,API,24835,37546,1.512,"\n joined, includes message framing"
9,gemini-3.1-pro-preview,API,24302,26099,1.074,\n joined
